In [1]:
%load_ext autoreload
%autoreload 2
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))
import warnings
warnings.filterwarnings("ignore")

import os
import pandas as pd
import sys
sys.path.insert(0, '/home/kat/Repos/SALSA/')

In [2]:
seedy = 666
df = pd.read_csv('data/chembl_valid_10to30atoms.csv')

samp_size = 100

df_samp = df.sample(samp_size, random_state=seedy)
df_samp['len'] = df_samp.smiles.apply(lambda x: len(x))
df_samp = df_samp.sort_values(by='len',ignore_index=True)

In [3]:
def get_augset_from_src(k,v,aug_iter):
        
    src_idx = v[0]
    anc_idx = v[1]
    smi = v[4]

    augs = get_ext_augs(smi)
    augs = list(set(augs))
    
    augset = []
    if augs:
        for aug_smi in augs:
            tup = [anc_idx, src_idx, aug_iter, aug_smi]
            if tup not in augset:
                augset.append(tup)
    return augset

In [4]:
import numpy as np
from tqdm.notebook import tqdm

from extended_augs_utils import get_ext_augs
from joblib import Parallel, delayed

iters = 5
max_augs = 10

for aug_iter in tqdm(range(iters+1), total=(iters+1)):
    
    curr_augset = []
    
    if aug_iter==0:
        for row in df_samp.itertuples():
            idx = row.Index
            anc_idx = idx
            src_idx = idx
            smi = row.smiles
            tup = [idx, anc_idx, src_idx, aug_iter, smi]
            curr_augset.append(tup)
            df_augset = pd.DataFrame(curr_augset, columns=['idx','anc_idx','src_idx', 
                                                           'aug_iter', 'smiles'])
        
    else:
        df_src = df_augset[df_augset.aug_iter==(aug_iter-1)]
        dict_src = df_src.T.to_dict('list')
            
        parallelizer = Parallel(n_jobs=-1, backend='multiprocessing' )
        tasks = (delayed(get_augset_from_src)(k,v,aug_iter) for k,v in dict_src.items())
        curr_augset = parallelizer(tasks)
        curr_augset = [x for l in curr_augset for x in l]
        curr_augset = pd.DataFrame(curr_augset, columns=['anc_idx', 'src_idx', 'aug_iter', 'smiles'])
        
        id_idc = [int(i) for i in range(len(df_src), len(curr_augset)+len(df_src))]
        print(len(id_idc))
        curr_augset['idx'] = id_idc
        curr_augset = curr_augset[['idx','anc_idx','src_idx', 'aug_iter', 'smiles']]
        curr_augset.drop_duplicates(keep='first', inplace=True)
        
        trunc_idc = []
        for i in range(len(df_samp)):
            df_anc = curr_augset[curr_augset.anc_idx==i]
            df_anc = df_anc.sample(n=10, replace=True)
            idc = df_anc.index.values
            trunc_idc.extend(idc)
        trunc_augset = curr_augset.iloc[trunc_idc]
        
        df_augset = pd.concat([df_augset, trunc_augset], axis=0)
        display(df_augset)

  0%|          | 0/6 [00:00<?, ?it/s]

930


,idx,anc_idx,src_idx,aug_iter,smiles
0,0,0,0,0,Cc1csc2nc(N)nn12
1,1,1,1,0,Cc1ncsc1C(O)c1ccccc1
2,2,2,2,0,Cc1cccc(C)c1NC1CCOC1=O
3,3,3,3,0,C#CCOc1ccc2ccc(=O)oc2c1
4,4,4,4,0,C#CC(=O)Nc1ccc(CCCCCC)cc1
...,...,...,...,...,...
923,1023,99,99,1,COc1cc(OC)c(C2CC(c3cc(C)cc([N+](=O)[O-])c3)=NN...
923,1023,99,99,1,COc1cc(OC)c(C2CC(c3cc(C)cc([N+](=O)[O-])c3)=NN...
926,1026,99,99,1,COc1cc(OC)c(C2CC(c3cccc([N+](=O)[O-])c3)=NN2C(...
929,1029,99,99,1,CCOc1cc(OC)c(C2CC(c3cccc([N+](=O)[O-])c3)=NN2C...


5973


,idx,anc_idx,src_idx,aug_iter,smiles
0,0,0,0,0,Cc1csc2nc(N)nn12
1,1,1,1,0,Cc1ncsc1C(O)c1ccccc1
2,2,2,2,0,Cc1cccc(C)c1NC1CCOC1=O
3,3,3,3,0,C#CCOc1ccc2ccc(=O)oc2c1
4,4,4,4,0,C#CC(=O)Nc1ccc(CCCCCC)cc1
...,...,...,...,...,...
5946,6946,99,1026,2,COc1cc(OC)c(C2CC(c3ccc(C)c([N+](=O)[O-])c3)=NN...
5941,6941,99,1022,2,CCC(=O)N1N=C(c2cccc([N+](=O)[O-])c2)CC1c1cc(OC...
5957,6957,99,1023,2,COc1cc(OC)c(C2CC(c3cc(C)cc([N+](=O)[O-])c3C)=N...
5960,6960,99,1023,2,CCc1cc(C2=NN(C(C)=O)C(c3cc(OC)c(OC)cc3OC)C2)cc...


...

,idx,anc_idx,src_idx,aug_iter,smiles
0,0,0,0,0,Cc1csc2nc(N)nn12
1,1,1,1,0,Cc1ncsc1C(O)c1ccccc1
2,2,2,2,0,Cc1cccc(C)c1NC1CCOC1=O
3,3,3,3,0,C#CCOc1ccc2ccc(=O)oc2c1
4,4,4,4,0,C#CC(=O)Nc1ccc(CCCCCC)cc1
...,...,...,...,...,...
8781,9781,99,6940,3,CCCC(=O)N1N=C(c2cc([N+](=O)[O-])ccc2C)CC1c1cc(...
8798,9798,99,6961,3,COc1cc(OC)c(C2CC(c3cc([N+](=O)[O-])c(C)c(C)c3C...
8866,9866,99,6928,3,CCOc1cc(OC)c(C2CC(c3cccc([N+](=O)[O-])c3C)=NN2...
8824,9824,99,6946,3,COc1cc(OC)c(C2CC(c3ccc(C)c([N+](=O)[O-])c3C)=N...


9172


,idx,anc_idx,src_idx,aug_iter,smiles
0,0,0,0,0,Cc1csc2nc(N)nn12
1,1,1,1,0,Cc1ncsc1C(O)c1ccccc1
2,2,2,2,0,Cc1cccc(C)c1NC1CCOC1=O
3,3,3,3,0,C#CCOc1ccc2ccc(=O)oc2c1
4,4,4,4,0,C#CC(=O)Nc1ccc(CCCCCC)cc1
...,...,...,...,...,...
9100,10100,99,9856,4,CCCc1cc(C2=NN(C(C)=O)C(c3c(OC)cc(OC)c(OC)c3C)C...
9092,10092,99,9856,4,CCCc1cc(C2=NN(C(C)=O)C(c3cc(OC)c(OCC)cc3OC)C2)...
9146,10146,99,9866,4,CCOc1cc(OC)c(C2CC(c3cccc([N+](=O)[O-])c3C)=NN2...
9086,10086,99,9802,4,CCOc1cc(OC)c(C2(CC)CC(c3cc(C)cc([N+](=O)[O-])c...


9199


,idx,anc_idx,src_idx,aug_iter,smiles
0,0,0,0,0,Cc1csc2nc(N)nn12
1,1,1,1,0,Cc1ncsc1C(O)c1ccccc1
2,2,2,2,0,Cc1cccc(C)c1NC1CCOC1=O
3,3,3,3,0,C#CCOc1ccc2ccc(=O)oc2c1
4,4,4,4,0,C#CC(=O)Nc1ccc(CCCCCC)cc1
...,...,...,...,...,...
9174,10174,99,10146,5,CCOc1cc(C2CC(c3cccc([N+](=O)[O-])c3C)=NN2C(=O)...
9155,10155,99,10100,5,CCCc1cc([N+](=O)[O-])cc(C2=NN(C(C)=O)C(c3c(OC)...
9113,10113,99,10162,5,CCOc1cc(OC)c(OC)cc1C1CC(c2cc([N+](=O)[O-])cc(C...
9189,10189,99,10109,5,CCOc1cc(OC)c(OC)cc1C1(C)C(C)C(c2cc(C)cc([N+](=...


In [6]:
df_augset_nodups = df_augset.drop_duplicates(keep='first')
df_augset_nodups

,idx,anc_idx,src_idx,aug_iter,smiles
0,0,0,0,0,Cc1csc2nc(N)nn12
1,1,1,1,0,Cc1ncsc1C(O)c1ccccc1
2,2,2,2,0,Cc1cccc(C)c1NC1CCOC1=O
3,3,3,3,0,C#CCOc1ccc2ccc(=O)oc2c1
4,4,4,4,0,C#CC(=O)Nc1ccc(CCCCCC)cc1
...,...,...,...,...,...
9174,10174,99,10146,5,CCOc1cc(C2CC(c3cccc([N+](=O)[O-])c3C)=NN2C(=O)...
9155,10155,99,10100,5,CCCc1cc([N+](=O)[O-])cc(C2=NN(C(C)=O)C(c3c(OC)...
9113,10113,99,10162,5,CCOc1cc(OC)c(OC)cc1C1CC(c2cc([N+](=O)[O-])cc(C...
9189,10189,99,10109,5,CCOc1cc(OC)c(OC)cc1C1(C)C(C)C(c2cc(C)cc([N+](=...


In [7]:
# df_augset_nodups.to_csv('20220603_extended_augset_1000anc_10iters.csv',index=False)